# Gurbani V2 Timestamps — Full Sri Guru Granth Sahib Ji Batch

This notebook recomputes the **V2 fixed timestamps** for every Ang (1–1430) on a free Colab GPU — the same pipeline that fixed Angs 1–8 and Ang 196 in the Gurbani Checker / Reader.

**How to run it**
1. Menu **Runtime → Change runtime type → T4 GPU** (if the first cell says GPU: NONE).
2. Menu **Runtime → Run all**.
3. When asked, allow the Google Drive mount. Finished files land in **MyDrive → gurbani-v2 → v2** (one `angN.v2.json` per Ang).
4. Keep this tab open while it runs. Progress prints after every Ang, with an ETA.

**If the session stops** (Colab time limit, disconnect, anything): just open the notebook again and **Run all** — Angs already finished are skipped automatically. Nothing is lost.

When it finishes (or whenever you want a partial hand-over), tell Muse — he will pull the folder from Drive and publish everything to the Checker + Reader.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — pehle Runtime > Change runtime type > T4 GPU chuno, phir Run all karo")

In [ ]:
# Dependencies (torch with GPU support is already in Colab)
!pip install -q transformers librosa ctc-forced-aligner soundfile huggingface_hub
print("installed")

In [ ]:
import os, re, urllib.request
os.makedirs("/content/gurbani-v2", exist_ok=True)
os.chdir("/content/gurbani-v2")
RAW = "https://raw.githubusercontent.com/KaramOfficial/gurbani-checker/main/pipeline/"
FILES = ["ctc_align.py", "chunk_align.py", "timestamper.py", "align_v2.py",
         "emit_only.py", "emit_gpu.py", "run_v2.py"]
for f in FILES:
    urllib.request.urlretrieve(RAW + f, f)
    print("got", f)
# timestamper.py imports faster_whisper but this pipeline never calls it — stub it
open("faster_whisper.py", "w").write(
    "class WhisperModel:\n    def __init__(self, *a, **k): raise RuntimeError('stub — not used in this pipeline')\n")
# Point the pipeline at a local download of the same Punjabi wav2vec2 model used for Angs 1-8/196
from huggingface_hub import snapshot_download
mp = snapshot_download("manandey/wav2vec2-large-xlsr-punjabi")
s = open("ctc_align.py", encoding="utf-8").read()
s2 = re.sub(r"MODEL_PATH = os\.expanduser\([\s\S]*?\)\n", 'MODEL_PATH = "' + mp + '"\n', s, count=1)
assert s2 != s, "MODEL_PATH patch failed — tell Muse"
open("ctc_align.py", "w", encoding="utf-8").write(s2)
print("pipeline ready · model:", mp)

In [ ]:
# Results live on YOUR Drive, so a stopped session loses nothing
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/gurbani-v2/v2"
os.makedirs(OUT, exist_ok=True)
START, END = 1, 1430   # full Sri Guru Granth Sahib Ji; shrink END for a quick test run
print("results ->", OUT)

In [ ]:
# The batch itself. Har Ang ton baad progress + ETA print hunda hai.
# Khatam hon te 'FINISHED' wali line dikhegi — phir Muse nu das deo.
!cd /content/gurbani-v2 && python run_v2.py --start {START} --end {END} --out-dir "{OUT}" --emitter emit_gpu.py

## Khatam hon ton baad
Muse nu ik message kar deo: **"Colab wala kamm ho gaya"** — oh Drive ton saariyaan files laike Checker te Reader vich publish kar devega, te live verify karke das devega.

Note: Ang 1–8 te 196 pehle hi live ne; eh batch ohna nu vi dobara bana ke bhej devega (koi nuksaan nahi — publish karan ton pehle Muse compare kar lainda hai).